# Machine Learning Prediction Model
## Renewable Hydrogen Production Forecasting

**Project:** Hydrogen Production Prediction using ML

**Dataset:** Renewable Hydrogen Production Data (2,535 samples)

**Objective:** Build and evaluate multiple ML models to predict hydrogen production rates

**Student:** AI & Data Science Engineering

**Date:** November 2025

---

## Key Predictions to Make:
1. **Hydrogen Production Rate (kg/day)** - Main target variable
2. **Feasibility Score** - Project viability classification
3. **System Efficiency** - Performance prediction

## Step 1: Install Libraries and Setup

In [ ]:
# Install required ML libraries
!pip install scikit-learn xgboost lightgbm catboost -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ML Libraries
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from xgboost import XGBRegressor
import xgboost as xgb

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("✓ All libraries imported successfully!")

## Step 2: Load and Prepare Data

In [ ]:
# Load cleaned dataset
df = pd.read_csv('Renewable-Hydrogen-Data.csv')

print("="*80)
print("DATASET INFORMATION")
print("="*80)
print(f"\nShape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values: {df.isnull().sum().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")

## Step 3: Exploratory Data Analysis

In [ ]:
# Correlation analysis for target variable: Hydrogen_Production_kg/day
print("\n" + "="*80)
print("CORRELATION WITH HYDROGEN PRODUCTION")
print("="*80)

numeric_cols = df.select_dtypes(include=[np.number]).columns
correlations = df[numeric_cols].corr()['Hydrogen_Production_kg/day'].sort_values(ascending=False)

print("\nTop Correlations:")
print(correlations)

In [ ]:
# Visualize correlations
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

top_features = correlations[1:7].index.tolist()  # Top 6 excluding self-correlation

for idx, feature in enumerate(top_features):
    ax = axes[idx // 3, idx % 3]
    ax.scatter(df[feature], df['Hydrogen_Production_kg/day'], alpha=0.5, s=20)
    ax.set_xlabel(feature, fontsize=10)
    ax.set_ylabel('Hydrogen Production (kg/day)', fontsize=10)
    corr_val = df[feature].corr(df['Hydrogen_Production_kg/day'])
    ax.set_title(f'{feature}\n(r = {corr_val:.3f})', fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Feature Correlations with Hydrogen Production', fontsize=14, fontweight='bold', y=1.002)
plt.show()

print("✓ Correlation plots generated!")

## Step 4: Data Preprocessing

In [ ]:
# Prepare data for modeling
print("\n" + "="*80)
print("DATA PREPROCESSING")
print("="*80)

# Create a copy
df_model = df.copy()

# Encode categorical variables
le = LabelEncoder()
df_model['Latitude_Band_Encoded'] = le.fit_transform(df_model['Latitude_Band'])

print(f"\nLatitude Band Encoding:")
for i, class_name in enumerate(le.classes_):
    print(f"  {i}: {class_name}")

# Define features and targets
feature_cols = ['Latitude', 'Longitude', 'Solar_Irradiance_kWh/m²/day', 
                'Wind_Speed_m/s', 'PV_Power_kW', 'Wind_Power_kW', 
                'System_Efficiency_%', 'Latitude_Band_Encoded']

X = df_model[feature_cols]
y_h2_production = df_model['Hydrogen_Production_kg/day']
y_feasibility = df_model['Feasibility_Score']
y_efficiency = df_model['System_Efficiency_%']

print(f"\nFeatures shape: {X.shape}")
print(f"Target (H2 Production) shape: {y_h2_production.shape}")
print(f"\nFeatures used: {feature_cols}")

In [ ]:
# Train-test split (80-20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_h2_production, test_size=0.2, random_state=42
)

print("\nTrain-Test Split:")
print(f"  Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"  Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)")

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Data preprocessing complete!")

## Step 5: Train Multiple ML Models

In [ ]:
print("\n" + "="*80)
print("BUILDING PREDICTION MODELS")
print("="*80)

# Dictionary to store models and results
models = {}
results = {}

# 1. Linear Regression
print("\n1. Linear Regression")
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
models['Linear Regression'] = lr_model
print("   ✓ Model trained")

# 2. Ridge Regression
print("\n2. Ridge Regression")
ridge_model = Ridge(alpha=1.0)
ridge_model.fit(X_train_scaled, y_train)
models['Ridge'] = ridge_model
print("   ✓ Model trained")

# 3. Lasso Regression
print("\n3. Lasso Regression")
lasso_model = Lasso(alpha=0.1)
lasso_model.fit(X_train_scaled, y_train)
models['Lasso'] = lasso_model
print("   ✓ Model trained")

# 4. Support Vector Regression
print("\n4. Support Vector Regression (SVR)")
svr_model = SVR(kernel='rbf', C=100, gamma='scale')
svr_model.fit(X_train_scaled, y_train)
models['SVR'] = svr_model
print("   ✓ Model trained")

# 5. Random Forest Regressor
print("\n5. Random Forest Regressor")
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
models['Random Forest'] = rf_model
print("   ✓ Model trained")

# 6. Gradient Boosting Regressor
print("\n6. Gradient Boosting Regressor")
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)
models['Gradient Boosting'] = gb_model
print("   ✓ Model trained")

# 7. XGBoost Regressor
print("\n7. XGBoost Regressor")
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train)
models['XGBoost'] = xgb_model
print("   ✓ Model trained")

print("\n" + "="*80)
print("✓ All 7 models trained successfully!")
print("="*80)

## Step 6: Model Evaluation & Predictions

In [ ]:
print("\n" + "="*80)
print("MODEL EVALUATION & PERFORMANCE METRICS")
print("="*80)

# Evaluate each model
evaluation_results = []

for model_name, model in models.items():
    # Make predictions
    if model_name in ['SVR']:
        y_pred_train = model.predict(X_train_scaled)
        y_pred_test = model.predict(X_test_scaled)
    else:
        if model_name in ['Linear Regression', 'Ridge', 'Lasso']:
            y_pred_train = model.predict(X_train_scaled)
            y_pred_test = model.predict(X_test_scaled)
        else:
            y_pred_train = model.predict(X_train)
            y_pred_test = model.predict(X_test)
    
    # Calculate metrics
    train_mse = mean_squared_error(y_train, y_pred_train)
    test_mse = mean_squared_error(y_test, y_pred_test)
    train_rmse = np.sqrt(train_mse)
    test_rmse = np.sqrt(test_mse)
    train_mae = mean_absolute_error(y_train, y_pred_train)
    test_mae = mean_absolute_error(y_test, y_pred_test)
    train_r2 = r2_score(y_train, y_pred_train)
    test_r2 = r2_score(y_test, y_pred_test)
    
    evaluation_results.append({
        'Model': model_name,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'Train R²': train_r2,
        'Test R²': test_r2
    })

# Create results dataframe
results_df = pd.DataFrame(evaluation_results)
print("\nModel Performance Comparison:\n")
print(results_df.to_string(index=False))

# Find best model
best_model_idx = results_df['Test R²'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'Model']
best_r2 = results_df.loc[best_model_idx, 'Test R²']

print(f"\n" + "="*80)
print(f"🏆 BEST MODEL: {best_model_name}")
print(f"Test R² Score: {best_r2:.4f}")
print("="*80)

## Step 7: Detailed Model Analysis

In [ ]:
# Get predictions from best model
best_model = models[best_model_name]

if best_model_name in ['SVR', 'Linear Regression', 'Ridge', 'Lasso']:
    y_pred_test_best = best_model.predict(X_test_scaled)
else:
    y_pred_test_best = best_model.predict(X_test)

print(f"\n" + "="*80)
print(f"DETAILED ANALYSIS: {best_model_name}")
print("="*80)

print(f"\nTest Set Metrics:")
print(f"  R² Score: {r2_score(y_test, y_pred_test_best):.4f}")
print(f"  RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test_best)):.4f} kg/day")
print(f"  MAE: {mean_absolute_error(y_test, y_pred_test_best):.4f} kg/day")
print(f"  Mean Absolute Percentage Error: {np.mean(np.abs((y_test - y_pred_test_best) / y_test)) * 100:.2f}%")

# Cross-validation score
if best_model_name in ['SVR', 'Linear Regression', 'Ridge', 'Lasso']:
    cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=5, scoring='r2')
else:
    cv_scores = cross_val_score(best_model, X_train, y_train, cv=5, scoring='r2')

print(f"\nCross-Validation Results (5-Fold):")
print(f"  CV Scores: {cv_scores}")
print(f"  Mean CV Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

In [ ]:
# Visualize predictions vs actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter plot: Predicted vs Actual
ax1 = axes[0]
ax1.scatter(y_test, y_pred_test_best, alpha=0.5, s=30)
min_val = min(y_test.min(), y_pred_test_best.min())
max_val = max(y_test.max(), y_pred_test_best.max())
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
ax1.set_xlabel('Actual H₂ Production (kg/day)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Predicted H₂ Production (kg/day)', fontsize=12, fontweight='bold')
ax1.set_title(f'{best_model_name}: Predicted vs Actual\nR² = {r2_score(y_test, y_pred_test_best):.4f}', 
              fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Residuals plot
ax2 = axes[1]
residuals = y_test - y_pred_test_best
ax2.scatter(y_pred_test_best, residuals, alpha=0.5, s=30)
ax2.axhline(y=0, color='r', linestyle='--', lw=2)
ax2.set_xlabel('Predicted H₂ Production (kg/day)', fontsize=12, fontweight='bold')
ax2.set_ylabel('Residuals (kg/day)', fontsize=12, fontweight='bold')
ax2.set_title('Residual Plot (Error Analysis)', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Prediction visualization generated!")

## Step 8: Feature Importance Analysis

In [ ]:
print("\n" + "="*80)
print("FEATURE IMPORTANCE ANALYSIS")
print("="*80)

if best_model_name in ['Random Forest', 'Gradient Boosting', 'XGBoost']:
    feature_importance = best_model.feature_importances_
    
    # Create importance dataframe
    importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': feature_importance
    }).sort_values('Importance', ascending=False)
    
    print("\nFeature Importance Ranking:\n")
    for idx, row in importance_df.iterrows():
        print(f"  {row['Feature']:<35} : {row['Importance']:.4f} ({row['Importance']*100:.2f}%)")
    
    # Visualize feature importance
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue')
    ax.set_xlabel('Importance', fontsize=12, fontweight='bold')
    ax.set_title(f'{best_model_name}: Feature Importance', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Feature importance plot generated!")
else:
    print("\nFeature importance not available for this model type.")
    print("(Available for tree-based models: Random Forest, Gradient Boosting, XGBoost)")

## Step 9: Sample Predictions on New Data

In [ ]:
print("\n" + "="*80)
print("SAMPLE PREDICTIONS ON NEW DATA")
print("="*80)

# Get 10 random test samples
sample_indices = np.random.choice(range(len(X_test)), 10, replace=False)
X_sample = X_test.iloc[sample_indices]
y_sample_actual = y_test.iloc[sample_indices]

# Make predictions
if best_model_name in ['SVR', 'Linear Regression', 'Ridge', 'Lasso']:
    X_sample_scaled = scaler.transform(X_sample)
    y_sample_pred = best_model.predict(X_sample_scaled)
else:
    y_sample_pred = best_model.predict(X_sample)

# Create prediction results dataframe
prediction_results = pd.DataFrame({
    'Actual H2 (kg/day)': y_sample_actual.values,
    'Predicted H2 (kg/day)': y_sample_pred,
    'Error (kg/day)': y_sample_actual.values - y_sample_pred,
    'Error %': ((y_sample_actual.values - y_sample_pred) / y_sample_actual.values * 100).round(2)
})

print("\nSample Predictions:\n")
print(prediction_results.to_string(index=False))

avg_error = np.abs(prediction_results['Error %']).mean()
print(f"\nAverage Prediction Error: {avg_error:.2f}%")
print(f"Accuracy: {100 - avg_error:.2f}%")

## Step 10: Model Comparison Summary

In [ ]:
print("\n" + "="*80)
print("COMPREHENSIVE MODEL COMPARISON")
print("="*80)

# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Test R² Score Comparison
ax1 = axes[0, 0]
results_sorted = results_df.sort_values('Test R²', ascending=True)
colors = ['green' if model == best_model_name else 'steelblue' for model in results_sorted['Model']]
ax1.barh(results_sorted['Model'], results_sorted['Test R²'], color=colors)
ax1.set_xlabel('R² Score', fontsize=11, fontweight='bold')
ax1.set_title('Model Performance: Test R² Score', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# 2. Test RMSE Comparison
ax2 = axes[0, 1]
results_sorted_rmse = results_df.sort_values('Test RMSE', ascending=True)
colors = ['green' if model == best_model_name else 'coral' for model in results_sorted_rmse['Model']]
ax2.barh(results_sorted_rmse['Model'], results_sorted_rmse['Test RMSE'], color=colors)
ax2.set_xlabel('RMSE (kg/day)', fontsize=11, fontweight='bold')
ax2.set_title('Model Performance: Test RMSE', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

# 3. Test MAE Comparison
ax3 = axes[1, 0]
results_sorted_mae = results_df.sort_values('Test MAE', ascending=True)
colors = ['green' if model == best_model_name else 'orange' for model in results_sorted_mae['Model']]
ax3.barh(results_sorted_mae['Model'], results_sorted_mae['Test MAE'], color=colors)
ax3.set_xlabel('MAE (kg/day)', fontsize=11, fontweight='bold')
ax3.set_title('Model Performance: Test MAE', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3, axis='x')

# 4. Overfitting Analysis
ax4 = axes[1, 1]
results_df['Overfitting'] = abs(results_df['Train R²'] - results_df['Test R²'])
colors = ['green' if model == best_model_name else 'mediumpurple' for model in results_df['Model']]
ax4.barh(results_df['Model'], results_df['Overfitting'], color=colors)
ax4.set_xlabel('R² Difference (Train - Test)', fontsize=11, fontweight='bold')
ax4.set_title('Overfitting Analysis (Lower is Better)', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("✓ Model comparison visualization generated!")

## Step 11: Save Model and Results

In [ ]:
import pickle

print("\n" + "="*80)
print("SAVING MODELS AND RESULTS")
print("="*80)

# Save best model
model_filename = f'{best_model_name.replace(" ", "_")}_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(best_model, f)
print(f"\n✓ Best model saved: {model_filename}")

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✓ Scaler saved: scaler.pkl")

# Save label encoder
with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)
print("✓ Label encoder saved: label_encoder.pkl")

# Save results
results_df.to_csv('model_evaluation_results.csv', index=False)
print("✓ Evaluation results saved: model_evaluation_results.csv")

print("\n" + "="*80)
print("All models and results saved successfully!")
print("="*80)

## Step 12: Final Summary Report

In [ ]:
print("\n" + "#"*80)
print("#" + " "*78 + "#")
print("#" + " "*20 + "MACHINE LEARNING MODEL SUMMARY REPORT" + " "*22 + "#")
print("#" + " "*78 + "#")
print("#"*80)

print("\n📊 PROJECT INFORMATION:")
print(f"  Dataset: Renewable Hydrogen Production Data")
print(f"  Total Samples: {len(df):,}")
print(f"  Training Samples: {len(X_train):,}")
print(f"  Test Samples: {len(X_test):,}")
print(f"  Features Used: {len(feature_cols)}")
print(f"  Target Variable: Hydrogen_Production_kg/day")

print("\n🤖 MODELS DEVELOPED:")
model_list = list(models.keys())
for i, model_name in enumerate(model_list, 1):
    marker = "🏆" if model_name == best_model_name else "  "
    print(f"  {marker} {i}. {model_name}")

print("\n🏆 BEST MODEL: {}".format(best_model_name))
best_row = results_df[results_df['Model'] == best_model_name].iloc[0]
print(f"  • Test R² Score: {best_row['Test R²']:.4f}")
print(f"  • Test RMSE: {best_row['Test RMSE']:.4f} kg/day")
print(f"  • Test MAE: {best_row['Test MAE']:.4f} kg/day")
print(f"  • Cross-Validation Score: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

print("\n📈 ACCURACY & PERFORMANCE:")
accuracy = (1 - (abs(y_test - y_pred_test_best).mean() / y_test.mean())) * 100
print(f"  • Average Accuracy: {accuracy:.2f}%")
print(f"  • Average Prediction Error: {(1 - accuracy):.2f}%")
print(f"  • Data Retention: 100% (no data lost in preprocessing)")

print("\n✅ DELIVERABLES:")
print(f"  ✓ 7 ML models trained and evaluated")
print(f"  ✓ Correlation analysis completed")
print(f"  ✓ Feature importance identified")
print(f"  ✓ Cross-validation performed (5-fold)")
print(f"  ✓ Prediction accuracy quantified")
print(f"  ✓ Best model saved for production")
print(f"  ✓ Results exported to CSV")

print("\n🎯 RECOMMENDATIONS:")
print(f"  1. Use {best_model_name} for hydrogen production predictions")
print(f"  2. Model achieves {best_row['Test R²']:.2%} variance explanation")
print(f"  3. Average prediction error: {abs(prediction_results['Error %']).mean():.2f}%")
print(f"  4. Suitable for real-world deployment")
print(f"  5. Monitor data distribution for drift detection")

print("\n" + "#"*80 + "\n")